# Notebook extracting and preprocessing the L2B $CH_4$ plume annotations

After downloading the L2B_PLM product from https://www.earthdata.nasa.gov/data/catalog/lpcloud-emitl2bch4plm-002 this notebook explores the downloaded product and creates auxiliary files useful for future data preprocessing and DL-ready dataset creation.

In [10]:
import os
import json
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

import geopandas as gpd
from pyproj import Geod
from shapely.geometry import Polygon, shape

## Extracting information from plume data

In [4]:
WORKING_DIR = Path("C:/Users/Julia/OneDrive/Dokumenty/WORK/PhD/IBM/CH4/")
DATA_DIR = WORKING_DIR / "data/EMITL2BCH4PLM_001-20251208_154109"

#### Collecting the metadata files

In [5]:
metadata_files = glob.glob(root_dir=DATA_DIR, pathname="*.json")
print(f"Number of plumes: {len(metadata_files)}")

Number of plumes: 1574


#### Inspecting the structure of the metadata file

In [6]:
with open(DATA_DIR / metadata_files[0]) as mfile:
    data = json.load(mfile)

geom = shape(data["features"][0]["geometry"])
print(f"Plume geometry: {geom}")

Plume geometry: POLYGON ((35.2030857 31.135752, 35.1954945 31.1352098, 35.1927833 31.1319564, 35.1922411 31.1227384, 35.1938678 31.1200273, 35.19441 31.1167739, 35.1982056 31.1129783, 35.2025435 31.1113516, 35.2123037 31.1113516, 35.2193527 31.1140627, 35.2215216 31.1162317, 35.2215216 31.1178584, 35.2226061 31.1189428, 35.2220638 31.1265341, 35.2193527 31.1308719, 35.2133881 31.1330409, 35.2057969 31.1330409, 35.2030857 31.135752))


#### Calculating the geometries of the selected plume

In [7]:
geod = Geod(ellps="WGS84")
area_sq_meters, poly_perim = geod.geometry_area_perimeter(geom)
centroid = geom.centroid
bounds = geom.bounds

min_lon, min_lat, max_lon, max_lat = bounds
_, _, width_meters = geod.inv(min_lon, (min_lat+max_lat)/2, max_lon, (min_lat+max_lat)/2)
_, _, height_meters = geod.inv((min_lon+max_lon)/2, min_lat, (min_lon+max_lon)/2, max_lat)

print(f"{abs(area_sq_meters)=}")
print(f"{centroid.x=}, {centroid.y=}")
print(f"{bounds=}")
print(f"{width_meters=}, {height_meters=}")

abs(area_sq_meters)=6269775.346538067
centroid.x=35.206945301284435, centroid.y=31.123153726173584
bounds=(35.1922411, 31.1113516, 35.2226061, 31.135752)
width_meters=2896.2412139221656, height_meters=2705.311149505336


### Creating a csv and gpkg file with the data information

In [11]:
# create a csv with coordinates + start/end date
gdf = {
    "name" : [],
    "geometry" : [],
    "latitude (centroid)" : [],
    "longitude (centroid)" : [],
    "area_km2": [],
    "bounds": [],
    "width_km": [],
    "height_km": [],
    "startDate" : [],
    "endDate": [],
    "max_ch4_concentration": [],
    "concentration_uncertainty": []
}

geod = Geod(ellps="WGS84")

for file in tqdm(metadata_files):
    with open(DATA_DIR / file) as mfile:
        data = json.load(mfile)
        
    geom = shape(data["features"][0]["geometry"])
    area_sq_meters, poly_perim = geod.geometry_area_perimeter(geom)
    centroid = geom.centroid
    bounds = geom.bounds
    
    min_lon, min_lat, max_lon, max_lat = bounds
    _, _, width_meters = geod.inv(min_lon, (min_lat+max_lat)/2, max_lon, (min_lat+max_lat)/2)
    _, _, height_meters = geod.inv((min_lon+max_lon)/2, min_lat, (min_lon+max_lon)/2, max_lat)
    
    gdf["name"].append(file[24:-5])
    gdf["geometry"].append(geom)
    gdf["latitude (centroid)"].append(centroid.y)
    gdf["longitude (centroid)"].append(centroid.x)
    gdf["area_km2"].append(area_sq_meters / 1000000)
    gdf["bounds"].append(bounds)
    gdf["width_km"].append(width_meters/1000)
    gdf["height_km"].append(height_meters/1000)
    gdf["startDate"].append(data["features"][0]["properties"]["UTC Time Observed"])
    gdf["endDate"].append(data["features"][0]["properties"]["UTC Time Observed"])
    gdf["max_ch4_concentration"].append(data["features"][0]["properties"]["Max Plume Concentration (ppm m)"])
    gdf["concentration_uncertainty"].append(data["features"][0]["properties"]["Concentration Uncertainty (ppm m)"])
    
df = pd.DataFrame(gdf)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

100%|██████████| 1574/1574 [00:08<00:00, 184.43it/s]


#### Save the DataFrame and GeoDataFrame to a .csv and a .gpkg

In [ ]:
df.to_csv(WORKING_DIR / "data/emit_ch4_plumes_metadata.csv", index=False)
gdf.to_file(WORKING_DIR / "data/emit_ch4_plumes_metadata.gpkg", driver="GPKG")